# Motion from Scratch

_How to go from pixels to motion._ Build a video engine from a single pixel up to a rebuilt motion-graphics intro, one layer at a time.

## How to use this notebook

1. **Save your own copy first:** File → Save a copy in Drive. Your code lives in this notebook, so the copy is how you keep your progress.
2. **Run the setup cell** below once per session. It downloads the course and checks that Node is available (Colab includes it).
3. **For each layer:** read the chapter, write your code in the cell that starts with `%%writefile`, run it, then run the test cell and the pictures cell.
4. **New session?** Run setup again, then re-run your `%%writefile` cells (Runtime → Run before) so your earlier layers are back on disk.

[Course home](https://nalediym.github.io/motion-from-scratch/) · [GitHub](https://github.com/nalediym/motion-from-scratch)

In [ ]:
#@title Setup: run once per session
import os, subprocess, glob
if not os.path.exists("motion-from-scratch") and not os.getcwd().endswith("motion-from-scratch"):
    subprocess.run(["git", "clone", "-q", "https://github.com/nalediym/motion-from-scratch"], check=True)
if not os.getcwd().endswith("motion-from-scratch"):
    os.chdir("motion-from-scratch")
node = subprocess.run(["node", "--version"], capture_output=True, text=True)
if node.returncode != 0:
    print("Installing Node...")
    subprocess.run("apt-get -qq update && apt-get -qq install -y nodejs > /dev/null", shell=True)
    node = subprocess.run(["node", "--version"], capture_output=True, text=True)
print("Course ready. Node", node.stdout.strip())

from IPython.display import Image, display
def run_js(code):
    """Run a Predict / Break it block against your code in src/."""
    open("scratch.js", "w").write(code)
    out = subprocess.run(["node", "tools/try.js", "--file", "scratch.js"], capture_output=True, text=True)
    print(out.stdout or out.stderr)

def pictures(layer):
    """Render a layer's test pictures with your code and show them."""
    out = subprocess.run(["node", "tools/pictures.js", layer], capture_output=True, text=True)
    print(out.stdout or out.stderr)
    for p in sorted(glob.glob(f"out/{layer}-*.png")):
        display(Image(p))

## Layer 00 · Pixels

_A framebuffer is memory you can see._

**You'll write:** `createFrame`, `pixelIndex`, `setPixel`, `getPixel`, `clearFrame` in `src/00-pixels.js`

Every frame of the KineticIntro you'll build in layer 09 is 1080 × 1920 pixels. Before anything can move, you need somewhere to put those pixels and a way to change them one at a time. That place is called a **framebuffer**, and by the end of this layer you'll have written one.

### Under the hood

#### A pixel is three or four numbers

Your screen is a grid of tiny lights. Each one mixes red, green and blue light, and how bright each color glows is a number from 0 (off) to 255 (full). Why 255? Each channel gets one **[byte](https://en.wikipedia.org/wiki/Byte)**: 8 bits, which can hold 2⁸ = 256 different values, 0 through 255.

```text
  [255,   0,   0]  red          [  0,   0,   0]  black (all off)
  [255, 255, 255]  white        [ 91, 140, 255]  Motion Studio's blue
```

A fourth number, **[alpha](https://en.wikipedia.org/wiki/Alpha_compositing)**, says how opaque the pixel is: 255 is solid, 0 is invisible. Alpha doesn't matter yet, but layer 01 is all about it. So one pixel is 4 bytes: **R, G, B, A**.

#### Memory is one long line

A picture is 2-D, but computer memory is a single long row of bytes. So you have to decide how to lay the grid out in a line. Nearly everything (browsers, image files, GPUs) uses the same answer: **row by row, left to right**, 4 bytes per pixel.

Here's a frame 4 pixels wide and 3 tall, with each cell showing where that pixel starts in the line:

```text
            x = 0   x = 1   x = 2   x = 3
          ┌───────┬───────┬───────┬───────┐
  y = 0   │   0   │   4   │   8   │  12   │
          ├───────┼───────┼───────┼───────┤
  y = 1   │  16   │  20   │  24   │  28   │
          ├───────┼───────┼───────┼───────┤
  y = 2   │  32   │  36   │  40   │  44   │
          └───────┴───────┴───────┴───────┘

  data: [ R G B A  R G B A  R G B A  R G B A | R G B A ... ]
          └ (0,0) ┘└ (1,0) ┘└ (2,0) ┘└ (3,0) ┘ └ (0,1) ...
          index 0   index 4   index 8   index 12  index 16
```

Look at the numbers for a while before writing `pixelIndex`. Going one step right adds the same amount each time. Going one step down adds a different, bigger amount. What is each amount, and why? If you can say that out loud, the formula writes itself.

> **Study buddy:** "Don't give me code. Ask me questions until I can explain what pixelIndex multiplies y by, and why there's a × 4." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2000%20%28Pixels%29%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Here%27s%20what%20I%20want%20help%20with%3A%20Don%27t%20give%20me%20code.%20Ask%20me%20questions%20until%20I%20can%20explain%20what%20pixelIndex%20multiplies%20y%20by%2C%20and%20why%20there%27s%20a%20%C3%97%204.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2000%20%28Pixels%29%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Here%27s%20what%20I%20want%20help%20with%3A%20Don%27t%20give%20me%20code.%20Ask%20me%20questions%20until%20I%20can%20explain%20what%20pixelIndex%20multiplies%20y%20by%2C%20and%20why%20there%27s%20a%20%C3%97%204.)

#### Why a Uint8ClampedArray

JavaScript has **[typed arrays](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Guide/Typed_arrays)**: fixed-size blocks of raw numbers, much faster than normal arrays because every slot is the same type. [`Uint8ClampedArray`](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/Uint8ClampedArray) means:

- **Uint8**: each slot is an unsigned 8-bit integer, 0–255. Exactly one channel.
- **Clamped**: if you store something out of range, it's pinned to the nearest edge instead of wrapping around. Store 300 and you get 255; store −5 and you get 0; store 127.6 and it rounds to 128. The exact rule is the [ToUint8Clamp](https://tc39.es/ecma262/#sec-touint8clamp) operation in the ECMAScript specification, the document that defines JavaScript.

The browser uses this exact type for the pixels of a canvas ([`ImageData.data`](https://developer.mozilla.org/en-US/docs/Web/API/ImageData/data)), which is why you're using it. The HTML standard [defines `ImageData`](https://html.spec.whatwg.org/multipage/imagebitmap-and-animations.html#the-imagedata-interface) as RGBA bytes stored left to right, row by row from the top: the same layout you're about to build. Compare it with its unclamped cousin [`Uint8Array`](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/Uint8Array), where 300 wraps around to 44 (300 − 256).

> **Study buddy:** "Quiz me on Uint8ClampedArray versus Uint8Array with three predict-the-output questions." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2000%20%28Pixels%29%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Here%27s%20what%20I%20want%20help%20with%3A%20Quiz%20me%20on%20Uint8ClampedArray%20versus%20Uint8Array%20with%20three%20predict-the-output%20questions.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2000%20%28Pixels%29%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Here%27s%20what%20I%20want%20help%20with%3A%20Quiz%20me%20on%20Uint8ClampedArray%20versus%20Uint8Array%20with%20three%20predict-the-output%20questions.)

#### The edge of the world

What should happen if someone calls `setPixel(frame, 4, 0, ...)` on a frame that's 4 pixels wide? There is no column 4. But the index formula will still happily compute a number, and that number lands somewhere real:

```text
  setPixel(frame, x = 4, y = 0)  on a 4-wide frame

          x = 0   x = 1   x = 2   x = 3   (x = 4?)
  y = 0   ░░░░░░  ░░░░░░  ░░░░░░  ░░░░░░  ← you meant here
  y = 1   ██████  ░░░░░░  ░░░░░░  ░░░░░░
          └─ but the formula points here: (0, 1)
```

This wrap-around bug is real: shapes that hang off the right edge reappear on the left, one row down. `setPixel` must check bounds itself, and later layers slide things off-screen all the time.

#### Getting it on screen

Your frame is just numbers in memory. To see it, `lib/present.js` (provided) wraps them in an [`ImageData`](https://developer.mozilla.org/en-US/docs/Web/API/ImageData) and copies them into a [`<canvas>`](https://developer.mozilla.org/en-US/docs/Web/API/Canvas_API) with [`putImageData`](https://developer.mozilla.org/en-US/docs/Web/API/CanvasRenderingContext2D/putImageData):

```text
  your frame.data ──► new ImageData(data, w, h) ──► ctx.putImageData(...)
        (CPU memory)        (browser wrapper)          │
                                                       ▼
                     display ◄── GPU ◄── compositor ◄── canvas
               (refreshes ~60 times a second)
```

You don't need to write that part. The important idea is that the screen only ever shows what's in memory, and drawing just means changing numbers.

### Your turn

Work through `src/00-pixels.js` top to bottom. Each function's comment is its spec. Some questions to guide you, in order:

1. **createFrame.** How many bytes does a `width × height` frame need? What does a brand-new typed array contain? What counts as a "positive whole number", and is `2.5` one? ([`Number.isInteger`](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/Number/isInteger) may help.)
2. **pixelIndex.** Use the grid above. What do you multiply `y` by to skip whole rows? Why the final `× 4`?
3. **setPixel.** What order do you do things in: round ([`Math.floor`](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/Math/floor)), check bounds, compute index, write? What happens if you compute the index before checking bounds? What should alpha be when the color only has three numbers?
4. **getPixel.** Why return a *new* array instead of a piece of `frame.data`? (Hint: what happens if the caller changes it?)
5. **clearFrame.** You could call `setPixel` for every pixel. That works. Is there a way that touches `frame.data` directly with a single loop? Which is faster, and why?

Run the tests after each function. A failing test tells you exactly which pixel was wrong.

Once it works, the test pictures show a checkerboard and a blue-and-orange XOR pattern.

> **Study buddy:** "Here's my setPixel. Don't fix it. Tell me which test case would break it, and let me find the bug." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2000%20%28Pixels%29%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Here%27s%20what%20I%20want%20help%20with%3A%20Here%27s%20my%20setPixel.%20Don%27t%20fix%20it.%20Tell%20me%20which%20test%20case%20would%20break%20it%2C%20and%20let%20me%20find%20the%20bug.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2000%20%28Pixels%29%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Here%27s%20what%20I%20want%20help%20with%3A%20Here%27s%20my%20setPixel.%20Don%27t%20fix%20it.%20Tell%20me%20which%20test%20case%20would%20break%20it%2C%20and%20let%20me%20find%20the%20bug.)



### ✏️ Your code for layer 00

Edit the cell below, then run it. The first line saves it as `src/00-pixels.js`. Then run the tests and pictures under it.

**Stuck?** Ask a study buddy that gives hints, not answers: [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2000%20%28Pixels%29%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Start%20by%20asking%20what%20I%27m%20working%20on.%20I%27ll%20paste%20my%20code%20and%20test%20output%20next.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2000%20%28Pixels%29%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Start%20by%20asking%20what%20I%27m%20working%20on.%20I%27ll%20paste%20my%20code%20and%20test%20output%20next.)

In [ ]:
%%writefile src/00-pixels.js
// Layer 00 · Pixels — a framebuffer: memory you can see.
// Read the chapter first (docs/00-pixels.md). Each comment below is a spec; the tests check it.
//
// A frame is a plain object:
//   { width, height, data }
// where `data` is a Uint8ClampedArray holding 4 numbers per pixel, row by row:
//
//   data:  [ R G B A | R G B A | R G B A | ... ]
//            pixel 0   pixel 1   pixel 2
//
// A color is an array [r, g, b] or [r, g, b, a], each 0–255.
// If alpha is missing, treat it as 255 (fully opaque).
(function (MS) {
  /**
   * Make a new frame, every pixel transparent black [0, 0, 0, 0].
   * Throw an Error if width or height is not a positive whole number.
   * @returns {{ width: number, height: number, data: Uint8ClampedArray }}
   */
  function createFrame(width, height) {
    throw MS.todo("createFrame");
  }

  /**
   * Where does pixel (x, y) start inside frame.data?
   * Return the index of its R value. (G, B and A follow right after it.)
   * Assume x and y are whole numbers inside the frame.
   */
  function pixelIndex(frame, x, y) {
    throw MS.todo("pixelIndex");
  }

  /**
   * Write `color` into pixel (x, y), replacing what was there.
   * - Round fractional coordinates down (2.7 → 2).
   * - If (x, y) is outside the frame, do nothing. Don't throw, and don't
   *   let x = width "wrap around" onto the next row.
   */
  function setPixel(frame, x, y, color) {
    throw MS.todo("setPixel");
  }

  /**
   * Read pixel (x, y) as a new array [r, g, b, a].
   * Round coordinates down like setPixel. Return null if outside the frame.
   */
  function getPixel(frame, x, y) {
    throw MS.todo("getPixel");
  }

  /**
   * Fill every pixel of the frame with `color`.
   */
  function clearFrame(frame, color) {
    throw MS.todo("clearFrame");
  }

  Object.assign(MS, { createFrame, pixelIndex, setPixel, getPixel, clearFrame });
})(globalThis.MS = globalThis.MS || {});


In [ ]:
# Tests for layer 00: run after saving your code above
!node tests/run.js 00

In [ ]:
# Pictures drawn by your code
pictures("00")

### Predict, then run

Guess before you run: the point is to catch the places where your mental model and the machine disagree.

**Predict 1** · Stuck? [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2000%20%28Pixels%29%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20prediction%3A%20%22Predict%201%3A%20how%20many%20bytes%20does%20one%20full-size%20video%20frame%20need%3F%22.%20Don%27t%20tell%20me%20the%20answer.%20Ask%20me%20how%20I%27m%20reasoning%20about%20it%2C%20and%20help%20me%20check%20my%20thinking.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2000%20%28Pixels%29%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20prediction%3A%20%22Predict%201%3A%20how%20many%20bytes%20does%20one%20full-size%20video%20frame%20need%3F%22.%20Don%27t%20tell%20me%20the%20answer.%20Ask%20me%20how%20I%27m%20reasoning%20about%20it%2C%20and%20help%20me%20check%20my%20thinking.)

In [ ]:
run_js(r"""
// Predict 1: how many bytes does one full-size video frame need?
const myGuess = null; // TODO: your guess, a single number
const frame = MS.createFrame(1080, 1920);
MS.reveal("data.length of a 1080 × 1920 frame", myGuess, frame.data.length);
""")

**Predict 2** · Stuck? [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2000%20%28Pixels%29%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20prediction%3A%20%22Predict%202%3A%20what%20comes%20back%20after%20writing%20out-of-range%20numbers%3F%22.%20Don%27t%20tell%20me%20the%20answer.%20Ask%20me%20how%20I%27m%20reasoning%20about%20it%2C%20and%20help%20me%20check%20my%20thinking.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2000%20%28Pixels%29%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20prediction%3A%20%22Predict%202%3A%20what%20comes%20back%20after%20writing%20out-of-range%20numbers%3F%22.%20Don%27t%20tell%20me%20the%20answer.%20Ask%20me%20how%20I%27m%20reasoning%20about%20it%2C%20and%20help%20me%20check%20my%20thinking.)

In [ ]:
run_js(r"""
// Predict 2: what comes back after writing out-of-range numbers?
const myGuess = null; // TODO: your guess as [r, g, b, a]
const f = MS.createFrame(1, 1);
MS.setPixel(f, 0, 0, [300, -5, 127.6]);
MS.reveal("getPixel after setPixel([300, -5, 127.6])", myGuess, MS.getPixel(f, 0, 0));
""")

**Predict 3** · Stuck? [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2000%20%28Pixels%29%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20prediction%3A%20%22Predict%203%3A%20what%20happens%20when%20you%20write%20to%20slot%201.5%3F%20%28None%20of%20your%20code%20is%20involved.%29%22.%20Don%27t%20tell%20me%20the%20answer.%20Ask%20me%20how%20I%27m%20reasoning%20about%20it%2C%20and%20help%20me%20check%20my%20thinking.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2000%20%28Pixels%29%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20prediction%3A%20%22Predict%203%3A%20what%20happens%20when%20you%20write%20to%20slot%201.5%3F%20%28None%20of%20your%20code%20is%20involved.%29%22.%20Don%27t%20tell%20me%20the%20answer.%20Ask%20me%20how%20I%27m%20reasoning%20about%20it%2C%20and%20help%20me%20check%20my%20thinking.)

In [ ]:
run_js(r"""
// Predict 3: what happens when you write to slot 1.5? (None of your code is involved.)
const myGuess = null; // TODO: your guess as an array of 4 numbers
const a = new Uint8ClampedArray(4);
a[1.5] = 200;
MS.reveal("the array after a[1.5] = 200", myGuess, a);
""")


<details><summary>Answers</summary>

1. 8,294,400 (1080 × 1920 × 4), about 8.3 MB. Six seconds at 30 fps is about 1.5 GB, yet the MP4 you export in layer 10 will be a few megabytes. Hold on to that gap; it's what video compression is for.
2. `[255, 0, 128, 255]`. Clamped, rounded, and alpha filled in.
3. `[0, 0, 0, 0]`. A typed array only has whole-number slots, and a write to a fractional index like `1.5` is silently thrown away: no error, no change. This is why `setPixel` must round coordinates. Forget it, and drawing at x = 2.5 quietly does nothing.

Bonus: try `a[0] = 127.5; a[1] = 128.5`. Both become 128. Clamped arrays round exact halves to the nearest even number ("banker's rounding"), so rounding errors don't all lean the same way.

</details>

### Break it

Once your tests pass, break things on purpose. Each **TODO** says what to change in your own code; run the experiment before and after, and explain the difference. Then undo the change and check your tests pass again.

#### 1. Swap x and y

**Break it 1** · Stuck? [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2000%20%28Pixels%29%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20break-it%20experiment%3A%20%22Break%20it%201%3A%20where%20do%20two%20pixels%20start%2C%20before%20and%20after%20the%20swap%3F%22.%20Don%27t%20tell%20me%20what%20will%20happen.%20Ask%20me%20what%20I%20observed%20before%20and%20after%20the%20change%2C%20and%20help%20me%20explain%20why.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2000%20%28Pixels%29%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20break-it%20experiment%3A%20%22Break%20it%201%3A%20where%20do%20two%20pixels%20start%2C%20before%20and%20after%20the%20swap%3F%22.%20Don%27t%20tell%20me%20what%20will%20happen.%20Ask%20me%20what%20I%20observed%20before%20and%20after%20the%20change%2C%20and%20help%20me%20explain%20why.)

In [ ]:
run_js(r"""
// Break it 1: where do two pixels start, before and after the swap?
// TODO(break-it), in src/00-pixels.js → pixelIndex:
//   swap x and y in your formula. Change nothing else.
const f = MS.createFrame(4, 2);   // 4 wide, 2 tall: data has 32 slots, 0 to 31
console.log("pixelIndex(3, 0) =", MS.pixelIndex(f, 3, 0));
console.log("pixelIndex(0, 1) =", MS.pixelIndex(f, 0, 1));
// TODO: note both numbers, make the change, run again.
// Is either new number outside the 32 slots? What would setPixel do with it?
""")


Then run the tests and look at the XOR pattern. Which tests catch the swap, and what happened to the picture?

#### 2. Delete the bounds check

**Break it 2** · Stuck? [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2000%20%28Pixels%29%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20break-it%20experiment%3A%20%22Break%20it%202%3A%20draw%20one%20pixel%20just%20past%20the%20right%20edge.%22.%20Don%27t%20tell%20me%20what%20will%20happen.%20Ask%20me%20what%20I%20observed%20before%20and%20after%20the%20change%2C%20and%20help%20me%20explain%20why.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2000%20%28Pixels%29%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20break-it%20experiment%3A%20%22Break%20it%202%3A%20draw%20one%20pixel%20just%20past%20the%20right%20edge.%22.%20Don%27t%20tell%20me%20what%20will%20happen.%20Ask%20me%20what%20I%20observed%20before%20and%20after%20the%20change%2C%20and%20help%20me%20explain%20why.)

In [ ]:
run_js(r"""
// Break it 2: draw one pixel just past the right edge.
// TODO(break-it), in src/00-pixels.js → setPixel:
//   comment out your check that (x, y) is inside the frame.
const f = MS.createFrame(4, 3);
MS.setPixel(f, 4, 0, [255, 255, 255]);   // x = width: there is no column 4
console.log(MS.asciiFrame(f));
// TODO: before the change you should see only dots (·).
// After it, which pixel lit up, and why that one?
""")


#### 3. Lose the clamping

**Break it 3** · Stuck? [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2000%20%28Pixels%29%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20break-it%20experiment%3A%20%22Break%20it%203%3A%20out-of-range%20values%2C%20with%20and%20without%20clamping.%22.%20Don%27t%20tell%20me%20what%20will%20happen.%20Ask%20me%20what%20I%20observed%20before%20and%20after%20the%20change%2C%20and%20help%20me%20explain%20why.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2000%20%28Pixels%29%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20break-it%20experiment%3A%20%22Break%20it%203%3A%20out-of-range%20values%2C%20with%20and%20without%20clamping.%22.%20Don%27t%20tell%20me%20what%20will%20happen.%20Ask%20me%20what%20I%20observed%20before%20and%20after%20the%20change%2C%20and%20help%20me%20explain%20why.)

In [ ]:
run_js(r"""
// Break it 3: out-of-range values, with and without clamping.
// TODO(break-it), in src/00-pixels.js → createFrame:
//   use Uint8Array instead of Uint8ClampedArray.
const f = MS.createFrame(1, 1);
MS.setPixel(f, 0, 0, [300, -5, 127.6]);
console.log(MS.getPixel(f, 0, 0));
// TODO: after the change, explain each of the three numbers.
// Hint: 256 possible values wrap around like a clock face.
""")


Which test fails with `Uint8Array`, and why would wrapping be a disaster for colors?

### Checkpoint

Your layer is **complete** when three things are true:

1. **All 11 tests pass.** That unlocks the checkpoint.
2. **The mystery picture comes out right.** The checkpoint runs your functions on a scrambled list of drawing commands full of this chapter's traps: fractional coordinates, pixels one step past the right edge, colors out of range. Get every rule right and a hidden word appears. Get one wrong and the picture shows it, with a hint about which rule to look at (never the fix).
3. **Three questions are answered correctly** in `checkpoints/00-pixels.js`. You may type an expression instead of a number, like `10 * 10`, or compute an answer with your own functions.

**Checkpoint 00** · Stuck? [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2000%20%28Pixels%29%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20working%20on%20this%20layer%27s%20checkpoint.%20Don%27t%20give%20me%20any%20answers%20or%20tell%20me%20if%20a%20specific%20answer%20is%20right.%20Ask%20me%20how%20I%27m%20working%20each%20question%20out%2C%20and%20help%20me%20reason%20through%20the%20mystery%20picture%27s%20hints.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2000%20%28Pixels%29%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20working%20on%20this%20layer%27s%20checkpoint.%20Don%27t%20give%20me%20any%20answers%20or%20tell%20me%20if%20a%20specific%20answer%20is%20right.%20Ask%20me%20how%20I%27m%20working%20each%20question%20out%2C%20and%20help%20me%20reason%20through%20the%20mystery%20picture%27s%20hints.)

In [ ]:
%%writefile checkpoints/00-pixels.js
// Checkpoint 00 · Pixels. Replace each null with your answer.
MS.answers("00", {
  // 1. In a frame 12 pixels wide, at which index of data does pixel (7, 3) start?
  q1: null,
  // 2. How many bytes do 90 frames of 1280 × 720 take, before any compression?
  q2: null,
  // 3. You store -12, 256, 2.5 and 3.5 into a new Uint8ClampedArray(4).
  //    What does it hold? Answer as an array of 4 numbers.
  q3: null,
});


In [ ]:
# Tests + checkpoint for layer 00
!node tests/run.js 00


### In the real world

- Video cards keep the image being shown in a region of memory literally called the framebuffer, scanned out to the display many times a second.
- Remotion doesn't touch pixels directly. It draws with the browser, then takes a screenshot of each frame, which is a full framebuffer like yours. You'll see that in layer 10.

### Read the originals

- [ECMAScript: TypedArray objects](https://tc39.es/ecma262/#sec-typedarray-objects): how typed arrays work, including why a write to index `1.5` is ignored.

### Going further (optional)

- Write `fillFrameFast(frame, color)` using [`Uint32Array`](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/Uint32Array) over the same [buffer](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/ArrayBuffer) (`new Uint32Array(frame.data.buffer)`) to set a whole pixel in one write. Time it against `clearFrame` on a 1080 × 1920 frame. Then look up **[endianness](https://en.wikipedia.org/wiki/Endianness)** to understand why the byte order inside that 32-bit number looks backwards.


## Layer 01 · Shapes & blending

_Deciding which pixels a shape covers, and what happens where shapes overlap._

**You'll write:** `hex`, `blendPixel`, `fillRect`, `fillCircle`, `mixColor`, `fillVerticalGradient`, `fillRadialGradient` in `src/01-shapes.js`
**Builds on:** layer 00

KineticIntro has a glowing radial background, words that fade in and out, and a light sweep that brightens the logo as it passes. All three are this layer: filling shapes, mixing colors, and **blending** a see-through color over what's already there.

### Under the hood

#### Colors as text: hex

Designers write colors like `#5B8CFF` (the CSS [hex color](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Values/hex-color) notation). That's three bytes written in base 16 ([hexadecimal](https://en.wikipedia.org/wiki/Hexadecimal)), two digits each:

```text
   #  5B   8C   FF
      │    │    └─ blue  = 0xFF = 15×16 + 15 = 255
      │    └────── green = 0x8C =  8×16 + 12 = 140
      └─────────── red   = 0x5B =  5×16 + 11 =  91
```

JavaScript can read base-16 text for you (look up [`parseInt`](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/parseInt)'s second argument, the radix). There are two common approaches: cut the string into three 2-character pieces, or read all six digits as one number and pull bytes out of it with bit shifts ([`>>`](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Operators/Right_shift)) and masks ([`& 255`](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Operators/Bitwise_AND)). Either is fine. The second shows you how colors are often packed in memory.

#### Rasterizing: which pixels does a shape cover?

A rectangle from x = 2 to x = 6 is an idea with perfect edges. The screen only has whole pixels. Turning shapes into pixels is called **[rasterization](https://en.wikipedia.org/wiki/Rasterisation)**, and the rule almost every renderer uses is: _a pixel belongs to the shape if its **center** is inside._

```text
        0     1     2     3     4     5     6     7
     ┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
     │  ·  │  ·  │  ■  │  ■  │  ■  │  ■  │  ·  │  ·  │   rect x = 2 .. 6
     └─────┴─────┴─────┴─────┴─────┴─────┴─────┴─────┘
                 ↑ centers at 2.5, 3.5, 4.5, 5.5 are inside
                                         ↑ 6.5 is not
```

For rectangles, testing centers is the same as rounding the edges, which is what the spec asks for.

The right edge is **exclusive** (6 is out). This "half-open" rule means two rectangles that share an edge (one from 2 to 6, the next from 6 to 9) meet perfectly, with no gap and no pixel painted twice. You'll see why that matters in **Break it**.

> **Study buddy:** "Ask me to predict which pixels fillRect(1.5, 0, 2, 1) covers, then tell me whether my reasoning (not just my answer) was right." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2001%20%28Shapes%20%26%20blending%29%3A%20rectangles%2C%20circles%2C%20alpha%2C%20gradients.%20Here%27s%20what%20I%20want%20help%20with%3A%20Ask%20me%20to%20predict%20which%20pixels%20fillRect%281.5%2C%200%2C%202%2C%201%29%20covers%2C%20then%20tell%20me%20whether%20my%20reasoning%20%28not%20just%20my%20answer%29%20was%20right.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2001%20%28Shapes%20%26%20blending%29%3A%20rectangles%2C%20circles%2C%20alpha%2C%20gradients.%20Here%27s%20what%20I%20want%20help%20with%3A%20Ask%20me%20to%20predict%20which%20pixels%20fillRect%281.5%2C%200%2C%202%2C%201%29%20covers%2C%20then%20tell%20me%20whether%20my%20reasoning%20%28not%20just%20my%20answer%29%20was%20right.)

#### Circles: distance from the center

A pixel is inside a circle if its center is no farther than `r` from the circle's center. That's Pythagoras: `dx² + dy² ≤ r²`. (You can skip the square root by comparing squared distances. Why does that give the same answer?)

```text
          · · · · · · · · ·
          · · · ■ ■ ■ · · ·         every ■ passes  dx² + dy² ≤ r²
          · · ■ ■ ■ ■ ■ · ·
          · · ■ ■ ● ■ ■ · ·         ● = center
          · · ■ ■ ■ ■ ■ · ·
          · · · ■ ■ ■ · · ·         ┌─────────┐  only test pixels
          · · · · · · · · ·         └─────────┘  inside this box
```

You *could* test every pixel in the frame. For a small circle on a 1080 × 1920 frame that's 2 million tests to draw maybe 300 pixels. Instead, only visit the circle's **bounding box**: from `cx − r` to `cx + r` in each direction.

Jagged edges like these are called **aliasing**. Real renderers soften them with **[antialiasing](https://en.wikipedia.org/wiki/Spatial_anti-aliasing)**: a pixel that's 40% covered gets 40% of the color. See **Going further**.

#### Alpha blending: painting with see-through paint

Here's the big one. When you paint a half-transparent red over a blue pixel, the result is a mix. When you paint it over an empty, fully transparent pixel, it should still be red, just see-through.

This was worked out in 1984 by Thomas Porter and Tom Duff at Lucasfilm, in a paper called ["Compositing Digital Images"](https://keithp.com/~keithp/porterduff/p253-porter.pdf) (SIGGRAPH '84, PDF). It's how film effects layered spaceships over backgrounds, and how every browser layers one element over another. Their "over" operation, with alphas written as 0 to 1:

```text
  a = alpha of the new color (source)     b = alpha of the existing pixel (destination)

  outAlpha = a + b × (1 − a)
  outRGB   = ( newRGB × a  +  oldRGB × b × (1 − a) ) / outAlpha
```

Read it as coverage. The new paint covers fraction `a` of the pixel. Of what's left, `(1 − a)`, the old pixel covers fraction `b`. The color is the average of the two, weighted by how much each one covers. The **divide by outAlpha** turns "amount of light" back into "color".

```text
  50% red over opaque blue:              50% red over nothing:
  outAlpha = 0.5 + 1 × 0.5 = 1           outAlpha = 0.5 + 0 = 0.5
  red  = (255×0.5 + 0) / 1   ≈ 128       red  = (255×0.5) / 0.5 = 255   ← still pure red
  blue = (0 + 255×1×0.5) / 1 ≈ 127       alpha = 0.5 → 128
```

Colors stored this way are called **straight alpha**. Many renderers store colors already multiplied by alpha instead (**premultiplied alpha**), because then the formula needs no division. Porter and Duff wrote "over" for premultiplied colors in the paper; the formula above is the same equation with the division put back so it works on straight colors. Browsers follow the [W3C Compositing and Blending spec](https://www.w3.org/TR/compositing-1/#porterduffcompositingoperators_srcover), which names this operator **source-over**.

> **Study buddy:** "Without giving me code, walk me through Porter–Duff 'over' with three cases I can check by hand, then explain straight versus premultiplied alpha with an analogy." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2001%20%28Shapes%20%26%20blending%29%3A%20rectangles%2C%20circles%2C%20alpha%2C%20gradients.%20Here%27s%20what%20I%20want%20help%20with%3A%20Without%20giving%20me%20code%2C%20walk%20me%20through%20Porter%E2%80%93Duff%20%27over%27%20with%20three%20cases%20I%20can%20check%20by%20hand%2C%20then%20explain%20straight%20versus%20premultiplied%20alpha%20with%20an%20analogy.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2001%20%28Shapes%20%26%20blending%29%3A%20rectangles%2C%20circles%2C%20alpha%2C%20gradients.%20Here%27s%20what%20I%20want%20help%20with%3A%20Without%20giving%20me%20code%2C%20walk%20me%20through%20Porter%E2%80%93Duff%20%27over%27%20with%20three%20cases%20I%20can%20check%20by%20hand%2C%20then%20explain%20straight%20versus%20premultiplied%20alpha%20with%20an%20analogy.)

#### Gradients: one color per position

A gradient answers one question per pixel: _how far along am I, from 0 to 1?_ Then it mixes the two colors by that amount:

```text
  mixColor(a, b, t) = a + (b − a) × t      (for each of R, G, B and A)

  t = 0          t = 0.25        t = 0.5         t = 0.75        t = 1
  ██ a           ██              ██ halfway      ██              ██ b
```

A vertical gradient gets `t` from the row. A radial gradient gets `t` from distance to a center point. Dividing each axis by its own radius (`rx`, `ry`) stretches the circle into an ellipse.

That's exactly what CSS does: [`radial-gradient()`](https://developer.mozilla.org/en-US/docs/Web/CSS/gradient/radial-gradient), defined in the [CSS Images spec](https://www.w3.org/TR/css-images-3/#radial-gradients), which says the first size is the horizontal radius and the second the vertical, each as a percentage of the box. KineticIntro's background is written in CSS as:

```text
  radial-gradient(70% 60% at 50% 38%, #1A2740, #0B0D12)
                  └rx  ry┘   └cx  cy┘   inner     outer
  on a 1080 × 1920 frame:  rx = 756   ry = 1152   cx = 540   cy = 729.6
```

### Your turn

1. **hex.** What should happen with `"#FFF"`, `"blue"`, or `"#12345G"`? A [regular expression](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Guide/Regular_expressions) like `/^#?[0-9a-f]{6}$/i` is one way to check. Can you explain each part of it?
2. **blendPixel.** Turn both alphas into 0–1 first. Handle the case where `outAlpha` is 0. What would dividing by 0 put in the pixel?
3. **fillRect.** Round the four edges, then clip them to the frame *before* looping. Why is clipping first better than letting `blendPixel` ignore off-screen pixels?
4. **fillCircle.** What does your bounding box look like when the circle hangs off the frame?
5. **mixColor, fillVerticalGradient, fillRadialGradient.** Where is the middle of row `y`? Why sample there instead of at its top edge?

Once it works, the test pictures show overlapping blended shapes and KineticIntro's exact background: the first real piece of the recreation.



### ✏️ Your code for layer 01

Edit the cell below, then run it. The first line saves it as `src/01-shapes.js`. Then run the tests and pictures under it.

**Stuck?** Ask a study buddy that gives hints, not answers: [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2001%20%28Shapes%20%26%20blending%29%3A%20rectangles%2C%20circles%2C%20alpha%2C%20gradients.%20Start%20by%20asking%20what%20I%27m%20working%20on.%20I%27ll%20paste%20my%20code%20and%20test%20output%20next.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2001%20%28Shapes%20%26%20blending%29%3A%20rectangles%2C%20circles%2C%20alpha%2C%20gradients.%20Start%20by%20asking%20what%20I%27m%20working%20on.%20I%27ll%20paste%20my%20code%20and%20test%20output%20next.)

In [ ]:
%%writefile src/01-shapes.js
// Layer 01 · Shapes & blending — rectangles, circles, alpha, gradients.
// Read the chapter first (docs/01-shapes.md). Each comment below is a spec; the tests check it.
// You can (and should) use your layer 00 functions here.
(function (MS) {
  /**
   * Turn a CSS-style hex string into a color array.
   *   hex("#5B8CFF")      → [91, 140, 255, 255]
   *   hex("5b8cff", 128)  → [91, 140, 255, 128]
   * Accept upper or lower case, with or without "#".
   * Throw an Error for anything that isn't exactly 6 hex digits.
   */
  function hex(str, alpha = 255) {
    throw MS.todo("hex");
  }

  /**
   * Paint `color` OVER pixel (x, y), respecting its alpha ("source-over").
   * With a = alpha of the new color, b = alpha of the existing pixel,
   * both as 0–1:
   *
   *   outAlpha = a + b × (1 − a)
   *   outRGB   = (newRGB × a + oldRGB × b × (1 − a)) / outAlpha
   *
   * Write outAlpha back as 0–255. If outAlpha is 0, the pixel becomes
   * [0, 0, 0, 0]. Round coordinates down; ignore pixels outside the frame.
   */
  function blendPixel(frame, x, y, color) {
    throw MS.todo("blendPixel");
  }

  /**
   * Blend a w × h rectangle whose top-left corner is (x, y).
   * Round the edges to whole pixels first:
   *   left = round(x), right = round(x + w)   (right is exclusive)
   *   top  = round(y), bottom = round(y + h)  (bottom is exclusive)
   * Clip to the frame, so a rectangle hanging off the edge is fine.
   */
  function fillRect(frame, x, y, w, h, color) {
    throw MS.todo("fillRect");
  }

  /**
   * Blend a circle with center (cx, cy) and radius r.
   * A pixel is inside when its CENTER, (px + 0.5, py + 0.5), is at
   * distance <= r from (cx, cy). Only visit pixels in the circle's
   * bounding box, not the whole frame.
   */
  function fillCircle(frame, cx, cy, r, color) {
    throw MS.todo("fillCircle");
  }

  /**
   * Mix two colors: t = 0 gives a, t = 1 gives b, 0.5 is halfway.
   * Mix all four channels (missing alpha = 255). Don't round.
   */
  function mixColor(a, b, t) {
    throw MS.todo("mixColor");
  }

  /**
   * Fill the whole frame from `top` color to `bottom` color.
   * Row y uses t = (y + 0.5) / height, so you sample each row at its middle.
   * Overwrite pixels (setPixel), don't blend.
   */
  function fillVerticalGradient(frame, top, bottom) {
    throw MS.todo("fillVerticalGradient");
  }

  /**
   * Fill the whole frame with an elliptical radial gradient, like CSS
   * `radial-gradient(rx ry at cx cy, inner, outer)`.
   * For each pixel center (x + 0.5, y + 0.5):
   *   d = sqrt( ((x + 0.5 − cx) / rx)² + ((y + 0.5 − cy) / ry)² )
   *   t = min(d, 1)
   * and write mixColor(inner, outer, t). Overwrite, don't blend.
   */
  function fillRadialGradient(frame, cx, cy, rx, ry, inner, outer) {
    throw MS.todo("fillRadialGradient");
  }

  Object.assign(MS, { hex, blendPixel, fillRect, fillCircle, mixColor, fillVerticalGradient, fillRadialGradient });
})(globalThis.MS = globalThis.MS || {});


In [ ]:
# Tests for layer 01: run after saving your code above
!node tests/run.js 01

In [ ]:
# Pictures drawn by your code
pictures("01")

### Predict, then run

Guess first, then run.

**Predict 1** · Stuck? [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2001%20%28Shapes%20%26%20blending%29%3A%20rectangles%2C%20circles%2C%20alpha%2C%20gradients.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20prediction%3A%20%22Predict%201%3A%20how%20many%20pixels%20does%20each%20rectangle%20cover%3F%22.%20Don%27t%20tell%20me%20the%20answer.%20Ask%20me%20how%20I%27m%20reasoning%20about%20it%2C%20and%20help%20me%20check%20my%20thinking.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2001%20%28Shapes%20%26%20blending%29%3A%20rectangles%2C%20circles%2C%20alpha%2C%20gradients.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20prediction%3A%20%22Predict%201%3A%20how%20many%20pixels%20does%20each%20rectangle%20cover%3F%22.%20Don%27t%20tell%20me%20the%20answer.%20Ask%20me%20how%20I%27m%20reasoning%20about%20it%2C%20and%20help%20me%20check%20my%20thinking.)

In [ ]:
run_js(r"""
// Predict 1: how many pixels does each rectangle cover?
const myGuess = null; // TODO: your guess as [first, second]
const count = f => { let n = 0; for (let i = 3; i < f.data.length; i += 4) if (f.data[i]) n++; return n; };
const a = MS.createFrame(5, 1), b = MS.createFrame(5, 1);
MS.fillRect(a, 0.4, 0, 2, 1, [255, 255, 255]);   // starts at x = 0.4
MS.fillRect(b, 0.6, 0, 2, 1, [255, 255, 255]);   // starts at x = 0.6
MS.reveal("pixels covered by each 2-wide rectangle", myGuess, [count(a), count(b)]);
""")

**Predict 2** · Stuck? [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2001%20%28Shapes%20%26%20blending%29%3A%20rectangles%2C%20circles%2C%20alpha%2C%20gradients.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20prediction%3A%20%22Predict%202%3A%20a%20circle%20of%20radius%205%20has%20area%20%CF%80r%C2%B2%20%E2%89%88%2078.5.%20How%20many%20pixels%20does%20it%20fill%3F%22.%20Don%27t%20tell%20me%20the%20answer.%20Ask%20me%20how%20I%27m%20reasoning%20about%20it%2C%20and%20help%20me%20check%20my%20thinking.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2001%20%28Shapes%20%26%20blending%29%3A%20rectangles%2C%20circles%2C%20alpha%2C%20gradients.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20prediction%3A%20%22Predict%202%3A%20a%20circle%20of%20radius%205%20has%20area%20%CF%80r%C2%B2%20%E2%89%88%2078.5.%20How%20many%20pixels%20does%20it%20fill%3F%22.%20Don%27t%20tell%20me%20the%20answer.%20Ask%20me%20how%20I%27m%20reasoning%20about%20it%2C%20and%20help%20me%20check%20my%20thinking.)

In [ ]:
run_js(r"""
// Predict 2: a circle of radius 5 has area πr² ≈ 78.5. How many pixels does it fill?
const myGuess = null; // TODO: your guess, a whole number
const f = MS.createFrame(21, 21);
MS.fillCircle(f, 10.5, 10.5, 5, [255, 255, 255]);
let n = 0;
for (let i = 3; i < f.data.length; i += 4) if (f.data[i]) n++;
MS.reveal("pixels in a radius-5 circle", myGuess, n);
""")

**Predict 3** · Stuck? [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2001%20%28Shapes%20%26%20blending%29%3A%20rectangles%2C%20circles%2C%20alpha%2C%20gradients.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20prediction%3A%20%22Predict%203%3A%20half-transparent%20red%20painted%20onto%20an%20empty%2C%20transparent%20pixel.%22.%20Don%27t%20tell%20me%20the%20answer.%20Ask%20me%20how%20I%27m%20reasoning%20about%20it%2C%20and%20help%20me%20check%20my%20thinking.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2001%20%28Shapes%20%26%20blending%29%3A%20rectangles%2C%20circles%2C%20alpha%2C%20gradients.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20prediction%3A%20%22Predict%203%3A%20half-transparent%20red%20painted%20onto%20an%20empty%2C%20transparent%20pixel.%22.%20Don%27t%20tell%20me%20the%20answer.%20Ask%20me%20how%20I%27m%20reasoning%20about%20it%2C%20and%20help%20me%20check%20my%20thinking.)

In [ ]:
run_js(r"""
// Predict 3: half-transparent red painted onto an empty, transparent pixel.
const myGuess = null; // TODO: your guess as [r, g, b, a]
const f = MS.createFrame(1, 1);
MS.blendPixel(f, 0, 0, [255, 0, 0, 128]);
MS.reveal("the pixel after blending", myGuess, MS.getPixel(f, 0, 0));
""")


<details><summary>Answers</summary>

1. `[2, 2]`. Edges round to 0 → 2 and 1 → 3.
2. 81. Close to 78.5 but not equal: pixels are all-or-nothing, so the count wobbles around the true area.
3. `[255, 0, 0, 128]`. The color stays pure red; only the alpha says "half there".

</details>

### Break it

Once your tests pass: make each TODO change, run the experiment before and after, explain the difference, then undo it.

#### 1. Forget the division

**Break it 1** · Stuck? [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2001%20%28Shapes%20%26%20blending%29%3A%20rectangles%2C%20circles%2C%20alpha%2C%20gradients.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20break-it%20experiment%3A%20%22Break%20it%201%3A%20blend%20onto%20nothing%2C%20then%20layer%20the%20result%20over%20white.%22.%20Don%27t%20tell%20me%20what%20will%20happen.%20Ask%20me%20what%20I%20observed%20before%20and%20after%20the%20change%2C%20and%20help%20me%20explain%20why.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2001%20%28Shapes%20%26%20blending%29%3A%20rectangles%2C%20circles%2C%20alpha%2C%20gradients.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20break-it%20experiment%3A%20%22Break%20it%201%3A%20blend%20onto%20nothing%2C%20then%20layer%20the%20result%20over%20white.%22.%20Don%27t%20tell%20me%20what%20will%20happen.%20Ask%20me%20what%20I%20observed%20before%20and%20after%20the%20change%2C%20and%20help%20me%20explain%20why.)

In [ ]:
run_js(r"""
// Break it 1: blend onto nothing, then layer the result over white.
// TODO(break-it), in src/01-shapes.js → blendPixel:
//   remove the "divide by outAlpha" step from the color math.
//   Keep the outAlpha math itself.
const f = MS.createFrame(1, 1);                  // transparent
MS.blendPixel(f, 0, 0, [255, 0, 0, 128]);        // 50% red onto nothing
const result = MS.getPixel(f, 0, 0);
console.log("50% red on nothing:", result);
const bg = MS.createFrame(1, 1);
MS.clearFrame(bg, [255, 255, 255]);              // white card
MS.blendPixel(bg, 0, 0, result);                 // put the result on top
console.log("...then over white:", MS.getPixel(bg, 0, 0));
// TODO: before the change the second line is a clean pink.
// After it, what color do you get, and why is it darker?
""")


Which tests fail? Dark halos around cut-out images in real software are this exact bug.

#### 2. Make the right edge inclusive

**Break it 2** · Stuck? [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2001%20%28Shapes%20%26%20blending%29%3A%20rectangles%2C%20circles%2C%20alpha%2C%20gradients.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20break-it%20experiment%3A%20%22Break%20it%202%3A%20two%20half-transparent%20rectangles%20that%20share%20an%20edge%20at%20x%20%3D%2010.%22.%20Don%27t%20tell%20me%20what%20will%20happen.%20Ask%20me%20what%20I%20observed%20before%20and%20after%20the%20change%2C%20and%20help%20me%20explain%20why.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2001%20%28Shapes%20%26%20blending%29%3A%20rectangles%2C%20circles%2C%20alpha%2C%20gradients.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20break-it%20experiment%3A%20%22Break%20it%202%3A%20two%20half-transparent%20rectangles%20that%20share%20an%20edge%20at%20x%20%3D%2010.%22.%20Don%27t%20tell%20me%20what%20will%20happen.%20Ask%20me%20what%20I%20observed%20before%20and%20after%20the%20change%2C%20and%20help%20me%20explain%20why.)

In [ ]:
run_js(r"""
// Break it 2: two half-transparent rectangles that share an edge at x = 10.
// TODO(break-it), in src/01-shapes.js → fillRect:
//   make the rectangle also cover the pixel at x = right.
const f = MS.createFrame(20, 1);
MS.clearFrame(f, [255, 255, 255]);
MS.fillRect(f, 0, 0, 10, 1, [0, 0, 255, 128]);   // covers 0..9
MS.fillRect(f, 10, 0, 10, 1, [0, 0, 255, 128]);  // covers 10..19
console.log("x = 9: ", MS.getPixel(f, 9, 0));
console.log("x = 10:", MS.getPixel(f, 10, 0));
console.log("x = 11:", MS.getPixel(f, 11, 0));
// TODO: before the change all three match. After it, what happens at x = 10?
// Why would that look like a seam?
""")


#### 3. Skip the bounding box

**Break it 3** · Stuck? [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2001%20%28Shapes%20%26%20blending%29%3A%20rectangles%2C%20circles%2C%20alpha%2C%20gradients.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20break-it%20experiment%3A%20%22Break%20it%203%3A%20how%20long%20do%2020%20small%20circles%20take%20on%20a%20full-size%20frame%3F%22.%20Don%27t%20tell%20me%20what%20will%20happen.%20Ask%20me%20what%20I%20observed%20before%20and%20after%20the%20change%2C%20and%20help%20me%20explain%20why.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2001%20%28Shapes%20%26%20blending%29%3A%20rectangles%2C%20circles%2C%20alpha%2C%20gradients.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20break-it%20experiment%3A%20%22Break%20it%203%3A%20how%20long%20do%2020%20small%20circles%20take%20on%20a%20full-size%20frame%3F%22.%20Don%27t%20tell%20me%20what%20will%20happen.%20Ask%20me%20what%20I%20observed%20before%20and%20after%20the%20change%2C%20and%20help%20me%20explain%20why.)

In [ ]:
run_js(r"""
// Break it 3: how long do 20 small circles take on a full-size frame?
// TODO(break-it), in src/01-shapes.js → fillCircle:
//   loop over every pixel in the frame instead of the circle's bounding box.
const f = MS.createFrame(1080, 1920);
const start = Date.now();
for (let i = 0; i < 20; i++) MS.fillCircle(f, 540, 960, 8, [255, 255, 255, 40]);
console.log("20 circles of radius 8:", Date.now() - start, "ms");
// TODO: run before and after. Roughly how many times slower is the full-frame loop?
// How many pixels does each version test per circle?
""")


### Checkpoint

When all 17 tests pass, the checkpoint unlocks (layer 00 explains how it works). This mystery picture's traps: squares that share edges, see-through ink painted onto nothing, then composited over a gradient. Then answer the three questions below.

**Checkpoint 01** · Stuck? [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2001%20%28Shapes%20%26%20blending%29%3A%20rectangles%2C%20circles%2C%20alpha%2C%20gradients.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20working%20on%20this%20layer%27s%20checkpoint.%20Don%27t%20give%20me%20any%20answers%20or%20tell%20me%20if%20a%20specific%20answer%20is%20right.%20Ask%20me%20how%20I%27m%20working%20each%20question%20out%2C%20and%20help%20me%20reason%20through%20the%20mystery%20picture%27s%20hints.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2001%20%28Shapes%20%26%20blending%29%3A%20rectangles%2C%20circles%2C%20alpha%2C%20gradients.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20working%20on%20this%20layer%27s%20checkpoint.%20Don%27t%20give%20me%20any%20answers%20or%20tell%20me%20if%20a%20specific%20answer%20is%20right.%20Ask%20me%20how%20I%27m%20working%20each%20question%20out%2C%20and%20help%20me%20reason%20through%20the%20mystery%20picture%27s%20hints.)

In [ ]:
%%writefile checkpoints/01-shapes.js
// Checkpoint 01 · Shapes & blending. Replace each null with your answer.
MS.answers("01", {
  // 1. You blend [255, 255, 255, 128] (half-transparent white) over an opaque black pixel.
  //    What is the pixel afterwards? Answer as [r, g, b, a].
  q1: null,
  // 2. A pixel is [0, 0, 255, 128]. You blend [255, 0, 0, 64] over it.
  //    What is its new alpha (0–255)?
  q2: null,
  // 3. How many pixels does fillRect(frame, 1.5, 0.5, 3, 2, color) cover?
  //    (Check what Math.round does with exact halves.)
  q3: null,
});


In [ ]:
# Tests + checkpoint for layer 01
!node tests/run.js 01


### In the real world

- Canvas's [`globalAlpha`](https://developer.mozilla.org/en-US/docs/Web/API/CanvasRenderingContext2D/globalAlpha) and CSS [`opacity`](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/opacity) both use Porter–Duff "over"; canvas exposes the full set of operators as [`globalCompositeOperation`](https://developer.mozilla.org/en-US/docs/Web/API/CanvasRenderingContext2D/globalCompositeOperation).
- Browsers usually keep pixels premultiplied internally. That's why a canvas pixel with very low alpha can come back from [`getImageData`](https://developer.mozilla.org/en-US/docs/Web/API/CanvasRenderingContext2D/getImageData) with slightly different colors than you put in: precision was lost in the multiply and divide.
- Motion Studio's `KineticIntro.tsx` paints its background with the CSS gradient above, then adds a second radial glow and a dark vignette on top. Those are three layers blended "over" each other, which you can now do by hand.

### Read the originals

- [Alvy Ray Smith, "Alpha and the History of Digital Compositing" (1995)](http://alvyray.com/Memos/CG/Microsoft/7_alpha.pdf): how the alpha channel was invented, by one of its inventors.

### Going further (optional)

- **Antialiased circles.** Instead of in-or-out, use coverage `clamp(r − distance + 0.5, 0, 1)` and multiply it into the color's alpha before blending. Compare the edges at scale in `tests.html`.
- **Premultiplied mode.** Write `blendPremultiplied` where colors are stored already multiplied by alpha. Show that the formula no longer needs a division, and find which test values it can't reproduce exactly after 8-bit rounding.


## Layer 02 · Glyphs

_A font is a lookup table from characters to tiny pictures._

**You'll write:** `drawGlyph`, `measureText`, `drawText` in `src/02-glyphs.js`
**Builds on:** layers 00–01 (especially `fillRect`)

KineticIntro is mostly words: **DESIGN.**, **ANIMATE.**, **RENDER.**, then **MOTION·STUDIO**. In this layer you'll draw text using nothing but the rectangles you already have and a font made of `#` and `.` characters. In layer 07 you'll switch to the browser's real fonts, and this layer is how you'll understand what the browser is doing for you.

### Under the hood

#### Two kinds of fonts

**[Bitmap fonts](https://en.wikipedia.org/wiki/Computer_font#Bitmap_fonts)** store each character as a grid of on/off pixels. Arcade machines, early terminals and the first Macintosh used them; [Susan Kare](https://en.wikipedia.org/wiki/Susan_Kare) drew the original Mac fonts on graph paper in 1983–84. They're simple and fast, but they only look right at one size.

**Outline fonts** ([TrueType](https://developer.apple.com/fonts/TrueType-Reference-Manual/), [OpenType](https://learn.microsoft.com/en-us/typography/opentype/spec/)) store each character as curves, so they can be drawn at any size. Drawing one means rasterizing the curves into pixels with antialiasing, which is the "which pixels does this shape cover?" problem from layer 01, done very well. That's what the browser does when you write `font-size: 150px`.

You're building a bitmap font renderer because it shows the whole pipeline with no magic: _character → lookup → grid → pixels_.

> **Study buddy:** "Explain how outline fonts get rasterized, connecting it to the pixel-center rule I learned in layer 01." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2002%20%28Glyphs%29%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Here%27s%20what%20I%20want%20help%20with%3A%20Explain%20how%20outline%20fonts%20get%20rasterized%2C%20connecting%20it%20to%20the%20pixel-center%20rule%20I%20learned%20in%20layer%2001.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2002%20%28Glyphs%29%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Here%27s%20what%20I%20want%20help%20with%3A%20Explain%20how%20outline%20fonts%20get%20rasterized%2C%20connecting%20it%20to%20the%20pixel-center%20rule%20I%20learned%20in%20layer%2001.)

#### The font file

`lib/font5x7.js` is the font. Every glyph is 7 rows of 5 cells:

```text
   MS.FONT_5x7.N  =               drawn at scale 3:
   [ "#...#",                     ■■■         ■■■
     "##..#",                     ■■■■■■      ■■■
     "#.#.#",                     ■■■   ■■■   ■■■
     "#..##",          ──►        ■■■      ■■■■■■
     "#...#",                     ■■■         ■■■
     "#...#",                     ■■■         ■■■
     "#...#" ]                    ■■■         ■■■
                                  (each # becomes a 3 × 3 block)
```

Reading it is a nested loop: for each row `r`, for each column `c`, if the cell is `#`, fill a `scale × scale` square at `(x + c × scale, y + r × scale)`. Scaling by repeating each pixel as a block like this is called **[nearest-neighbor](https://en.wikipedia.org/wiki/Nearest-neighbor_interpolation)** scaling. It's why pixel art stays crisp.

#### Laying out a line

Text layout is the job of deciding where each glyph goes. Ours is the simplest possible version:

```text
   x                                                    one glyph = 5 cells
   │◄─ 5 ─►│◄1►│◄─ 5 ─►│◄1►│◄─ 5 ─►│                    tracking  = gap between glyphs
   ┌───────┐   ┌───────┐   ┌───────┐
   │   H   │   │   I   │   │   !   │                    width = 3 × 5 + 2 × 1 = 17 cells
   └───────┘   └───────┘   └───────┘                          (× scale for pixels)
   ◄──────────── measureText = 17 ───────────►
```

- **Advance** is how far you move right after each glyph: its width plus the tracking.
- **Tracking** is uniform extra space *between* glyphs, so none after the last one (CSS calls it [`letter-spacing`](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/letter-spacing)). [Kerning](https://en.wikipedia.org/wiki/Kerning), which tightens specific pairs like "AV", is different; our font skips it.

**Alignment needs measuring first.** To center text on `x`, you need its total width before drawing the first glyph, so the left edge is `x − width / 2`. Real layout engines work in the same two passes: measure everything, then place everything. KineticIntro centers every word, so you'll use this constantly.

> **Study buddy:** "Ask me questions until I can explain the difference between advance, tracking and kerning. No code." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2002%20%28Glyphs%29%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Here%27s%20what%20I%20want%20help%20with%3A%20Ask%20me%20questions%20until%20I%20can%20explain%20the%20difference%20between%20advance%2C%20tracking%20and%20kerning.%20No%20code.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2002%20%28Glyphs%29%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Here%27s%20what%20I%20want%20help%20with%3A%20Ask%20me%20questions%20until%20I%20can%20explain%20the%20difference%20between%20advance%2C%20tracking%20and%20kerning.%20No%20code.)

#### Characters are trickier than they look

- **Case.** The font has only uppercase. Mapping `"g"` to `"G"` (with [`toUpperCase`](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/String/toUpperCase)) is a tiny case of what real fonts do with **fallback**: when the font lacks a character, find a substitute.
- **Missing characters.** When there's no substitute, show **tofu**: an empty box. (Google's font family [Noto](https://fonts.google.com/noto) is named for "no more tofu".) Tofu is ugly on purpose. It tells you something is missing instead of silently skipping it.
- **What counts as one character?** `"MOTION·STUDIO"` has 13 characters, and `·` is one of them. But try `"👋".length` in the console: it says 2. JavaScript strings count 16-bit units ([`String.length`](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/String/length)), and characters beyond the first 65,536 take two, called a [surrogate pair](https://www.unicode.org/faq/utf_bom.html) in Unicode's UTF-16 encoding. [`for...of`](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Statements/for...of) and [spread](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Operators/Spread_syntax) (`[...text]`) walk real characters. A plain `for (let i = 0; i < text.length; i++)` splits that emoji in half. Pick your loop carefully.

#### Sizing for the real video

KineticIntro sets its words at a font size of 150 px in a 1080 px wide frame. Font size is the height of the font's whole design box (the "em"), not of a capital letter ([`font-size`](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/font-size)). In Inter, the font it uses, capitals are about 73% of that, so roughly 110 px tall. Our capitals are 7 cells tall, so a scale of about 16 matches. Can **ANIMATE.** fit across 1080 pixels at scale 16 with tracking 1? How large can the scale get before it doesn't? Work it out with `measureText` once it passes.

### Your turn

1. **measureText.** Start here: it's pure arithmetic and has the clearest tests. What should it return for the empty string? For one glyph?
2. **drawGlyph.** Uppercase first, and tofu if it's still missing. The loop is in *The font file* above.
3. **drawText.** Measure once, work out the starting x from `align`, then walk the characters, advancing by one glyph plus tracking each time.

Once it works, the test pictures show **DESIGN.** in big pixel letters, plus an alignment test with a pink center line and one tofu box.

> **Study buddy:** "Here's my measureText. Don't fix it; give me one input where it's wrong and let me work out why." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2002%20%28Glyphs%29%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Here%27s%20what%20I%20want%20help%20with%3A%20Here%27s%20my%20measureText.%20Don%27t%20fix%20it%3B%20give%20me%20one%20input%20where%20it%27s%20wrong%20and%20let%20me%20work%20out%20why.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2002%20%28Glyphs%29%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Here%27s%20what%20I%20want%20help%20with%3A%20Here%27s%20my%20measureText.%20Don%27t%20fix%20it%3B%20give%20me%20one%20input%20where%20it%27s%20wrong%20and%20let%20me%20work%20out%20why.)



### ✏️ Your code for layer 02

Edit the cell below, then run it. The first line saves it as `src/02-glyphs.js`. Then run the tests and pictures under it.

**Stuck?** Ask a study buddy that gives hints, not answers: [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2002%20%28Glyphs%29%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Start%20by%20asking%20what%20I%27m%20working%20on.%20I%27ll%20paste%20my%20code%20and%20test%20output%20next.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2002%20%28Glyphs%29%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Start%20by%20asking%20what%20I%27m%20working%20on.%20I%27ll%20paste%20my%20code%20and%20test%20output%20next.)

In [ ]:
%%writefile src/02-glyphs.js
// Layer 02 · Glyphs — text from a hand-made bitmap font.
// Read the chapter first (docs/02-glyphs.md). Each comment below is a spec; the tests check it.
//
// The font lives in lib/font5x7.js as MS.FONT_5x7: an object mapping a
// character to 7 strings of 5 characters, "#" for ink and "." for empty.
// Draw with fillRect from layer 01.
(function (MS) {
  const GLYPH_W = 5;
  const GLYPH_H = 7;

  /**
   * Draw one character with its top-left corner at (x, y).
   * Each "#" cell becomes a filled square of size `scale` × `scale`.
   * - The font only has uppercase letters: "a" must draw like "A".
   * - A character missing from the font draws "tofu": a hollow 5 × 7 box
   *   (the outer ring of cells filled, the inside empty).
   */
  function drawGlyph(frame, ch, x, y, scale, color) {
    throw MS.todo("drawGlyph");
  }

  /**
   * How wide is `text` in pixels?
   * Each glyph is GLYPH_W cells wide, and `tracking` empty cells sit
   * BETWEEN glyphs (not after the last one). Multiply cells by `scale`.
   *   measureText("AB", 1)    → 5 + 1 + 5 = 11
   *   measureText("AB", 3)    → 33
   *   measureText("", 3)      → 0
   */
  function measureText(text, scale, tracking = 1) {
    throw MS.todo("measureText");
  }

  /**
   * Draw a line of text and return its width (same number measureText gives).
   * opts.tracking : empty cells between glyphs (default 1)
   * opts.align    : "left" (default) — x is the left edge
   *                 "center"         — x is the middle
   *                 "right"          — x is the right edge
   * A space draws nothing but still takes up a glyph's width.
   */
  function drawText(frame, text, x, y, scale, color, opts = {}) {
    throw MS.todo("drawText");
  }

  Object.assign(MS, { GLYPH_W, GLYPH_H, drawGlyph, measureText, drawText });
})(globalThis.MS = globalThis.MS || {});


In [ ]:
# Tests for layer 02: run after saving your code above
!node tests/run.js 02

In [ ]:
# Pictures drawn by your code
pictures("02")

### Predict, then run

Guess first, then run.

**Predict 1** · Stuck? [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2002%20%28Glyphs%29%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20prediction%3A%20%22Predict%201%3A%20how%20wide%20is%20%22DESIGN.%22%20at%20scale%204%20with%20tracking%201%3F%22.%20Don%27t%20tell%20me%20the%20answer.%20Ask%20me%20how%20I%27m%20reasoning%20about%20it%2C%20and%20help%20me%20check%20my%20thinking.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2002%20%28Glyphs%29%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20prediction%3A%20%22Predict%201%3A%20how%20wide%20is%20%22DESIGN.%22%20at%20scale%204%20with%20tracking%201%3F%22.%20Don%27t%20tell%20me%20the%20answer.%20Ask%20me%20how%20I%27m%20reasoning%20about%20it%2C%20and%20help%20me%20check%20my%20thinking.)

In [ ]:
run_js(r"""
// Predict 1: how wide is "DESIGN." at scale 4 with tracking 1?
const myGuess = null; // TODO: your guess in pixels
MS.reveal('measureText("DESIGN.", 4)', myGuess, MS.measureText("DESIGN.", 4));
""")

**Predict 2** · Stuck? [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2002%20%28Glyphs%29%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20prediction%3A%20%22Predict%202%3A%20%22%C3%84%22%20isn%27t%20in%20the%20font.%20How%20many%20of%20its%2035%20cells%20get%20ink%3F%22.%20Don%27t%20tell%20me%20the%20answer.%20Ask%20me%20how%20I%27m%20reasoning%20about%20it%2C%20and%20help%20me%20check%20my%20thinking.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2002%20%28Glyphs%29%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20prediction%3A%20%22Predict%202%3A%20%22%C3%84%22%20isn%27t%20in%20the%20font.%20How%20many%20of%20its%2035%20cells%20get%20ink%3F%22.%20Don%27t%20tell%20me%20the%20answer.%20Ask%20me%20how%20I%27m%20reasoning%20about%20it%2C%20and%20help%20me%20check%20my%20thinking.)

In [ ]:
run_js(r"""
// Predict 2: "Ä" isn't in the font. How many of its 35 cells get ink?
const myGuess = null; // TODO: a number from 0 to 35
const f = MS.createFrame(5, 7);
MS.drawGlyph(f, "Ä", 0, 0, 1, [255, 255, 255]);
let n = 0;
for (let i = 3; i < f.data.length; i += 4) if (f.data[i]) n++;
MS.reveal("cells inked for Ä", myGuess, n);
console.log(MS.asciiFrame(f));
""")

**Predict 3** · Stuck? [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2002%20%28Glyphs%29%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20prediction%3A%20%22Predict%203%3A%20centered%20on%20x%20%3D%2020%2C%20%22II%22%20is%2011%20wide%2C%20so%20its%20left%20edge%20is%2014.5.%22.%20Don%27t%20tell%20me%20the%20answer.%20Ask%20me%20how%20I%27m%20reasoning%20about%20it%2C%20and%20help%20me%20check%20my%20thinking.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2002%20%28Glyphs%29%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20prediction%3A%20%22Predict%203%3A%20centered%20on%20x%20%3D%2020%2C%20%22II%22%20is%2011%20wide%2C%20so%20its%20left%20edge%20is%2014.5.%22.%20Don%27t%20tell%20me%20the%20answer.%20Ask%20me%20how%20I%27m%20reasoning%20about%20it%2C%20and%20help%20me%20check%20my%20thinking.)

In [ ]:
run_js(r"""
// Predict 3: centered on x = 20, "II" is 11 wide, so its left edge is 14.5.
// Which column gets the first ink?
const myGuess = null; // TODO: a column number
const f = MS.createFrame(40, 7);
MS.drawText(f, "II", 20, 0, 1, [255, 255, 255], { align: "center" });
let first = -1;
for (let x = 0; x < f.width; x++) if (MS.getPixel(f, x, 0)[3]) { first = x; break; }
MS.reveal("first inked column", myGuess, first);
""")


<details><summary>Answers</summary>

1. 164 pixels: 7 glyphs × 5 cells × 4, plus 6 gaps × 1 × 4.
2. 20. "Ä" isn't in the font and uppercasing it doesn't help, so you get tofu: the 20 cells around the edge of the box.
3. Column 15. Your `fillRect` rounds edges, and `round(14.5)` is 15. Text lands on whole pixels, just like rectangles.

</details>

### Break it

Once your tests pass: make each TODO change, run the experiment before and after, explain the difference, then undo it.

#### 1. Tracking after every glyph

**Break it 1** · Stuck? [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2002%20%28Glyphs%29%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20break-it%20experiment%3A%20%22Break%20it%201%3A%20is%20centered%20text%20still%20centered%3F%22.%20Don%27t%20tell%20me%20what%20will%20happen.%20Ask%20me%20what%20I%20observed%20before%20and%20after%20the%20change%2C%20and%20help%20me%20explain%20why.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2002%20%28Glyphs%29%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20break-it%20experiment%3A%20%22Break%20it%201%3A%20is%20centered%20text%20still%20centered%3F%22.%20Don%27t%20tell%20me%20what%20will%20happen.%20Ask%20me%20what%20I%20observed%20before%20and%20after%20the%20change%2C%20and%20help%20me%20explain%20why.)

In [ ]:
run_js(r"""
// Break it 1: is centered text still centered?
// TODO(break-it), in src/02-glyphs.js → measureText:
//   count tracking after every glyph, including the last one.
const f = MS.createFrame(40, 7);
MS.drawText(f, "II", 20, 0, 1, [255, 255, 255], { align: "center" });
let left = -1, right = -1;
for (let x = 0; x < f.width; x++) if (MS.getPixel(f, x, 0)[3]) { if (left < 0) left = x; right = x; }
console.log("empty columns on the left:", left, "| on the right:", f.width - 1 - right);
console.log(MS.asciiFrame(f));
// TODO: before the change the two sides differ by one column (from rounding).
// After it, which way does the text shift, and by how much? Why that direction?
""")


Real UI text that looks "slightly off-center" is often this bug.

#### 2. Loop by index

**Break it 2** · Stuck? [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2002%20%28Glyphs%29%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20break-it%20experiment%3A%20%22Break%20it%202%3A%20one%20emoji%2C%20drawn%20by%20your%20drawText.%22.%20Don%27t%20tell%20me%20what%20will%20happen.%20Ask%20me%20what%20I%20observed%20before%20and%20after%20the%20change%2C%20and%20help%20me%20explain%20why.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2002%20%28Glyphs%29%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20break-it%20experiment%3A%20%22Break%20it%202%3A%20one%20emoji%2C%20drawn%20by%20your%20drawText.%22.%20Don%27t%20tell%20me%20what%20will%20happen.%20Ask%20me%20what%20I%20observed%20before%20and%20after%20the%20change%2C%20and%20help%20me%20explain%20why.)

In [ ]:
run_js(r"""
// Break it 2: one emoji, drawn by your drawText.
// TODO(break-it), in src/02-glyphs.js → drawText:
//   walk the text with for (let i = 0; i < text.length; i++) and text[i].
console.log('"HI 👋".length =', "HI 👋".length, '  [..."HI 👋"].length =', [..."HI 👋"].length);
const f = MS.createFrame(30, 7);
MS.drawText(f, "HI 👋", 0, 0, 1, [255, 255, 255]);
console.log(MS.asciiFrame(f));
// TODO: count the tofu boxes before and after the change. Why does one emoji become two?
""")


#### 3. Skip the uppercase step

**Break it 3** · Stuck? [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2002%20%28Glyphs%29%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20break-it%20experiment%3A%20%22Break%20it%203%3A%20mixed-case%20text.%22.%20Don%27t%20tell%20me%20what%20will%20happen.%20Ask%20me%20what%20I%20observed%20before%20and%20after%20the%20change%2C%20and%20help%20me%20explain%20why.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2002%20%28Glyphs%29%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20doing%20this%20break-it%20experiment%3A%20%22Break%20it%203%3A%20mixed-case%20text.%22.%20Don%27t%20tell%20me%20what%20will%20happen.%20Ask%20me%20what%20I%20observed%20before%20and%20after%20the%20change%2C%20and%20help%20me%20explain%20why.)

In [ ]:
run_js(r"""
// Break it 3: mixed-case text.
// TODO(break-it), in src/02-glyphs.js → drawGlyph:
//   look the character up exactly as given, without uppercasing it.
const f = MS.createFrame(36, 7);
MS.drawText(f, "Design", 0, 0, 1, [255, 255, 255]);
console.log(MS.asciiFrame(f));
// TODO: predict what you'll see after the change, then run. Which letters survive?
""")


### Checkpoint

When all 11 tests pass, the checkpoint unlocks (layer 00 explains how it works). This mystery picture's traps: lowercase text, an emoji, middle dots, and three lines that must all be centered. Then answer the three questions below.

**Checkpoint 02** · Stuck? [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2002%20%28Glyphs%29%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20working%20on%20this%20layer%27s%20checkpoint.%20Don%27t%20give%20me%20any%20answers%20or%20tell%20me%20if%20a%20specific%20answer%20is%20right.%20Ask%20me%20how%20I%27m%20working%20each%20question%20out%2C%20and%20help%20me%20reason%20through%20the%20mystery%20picture%27s%20hints.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2002%20%28Glyphs%29%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27m%20working%20on%20this%20layer%27s%20checkpoint.%20Don%27t%20give%20me%20any%20answers%20or%20tell%20me%20if%20a%20specific%20answer%20is%20right.%20Ask%20me%20how%20I%27m%20working%20each%20question%20out%2C%20and%20help%20me%20reason%20through%20the%20mystery%20picture%27s%20hints.)

In [ ]:
%%writefile checkpoints/02-glyphs.js
// Checkpoint 02 · Glyphs. Replace each null with your answer.
MS.answers("02", {
  // 1. How wide, in pixels, is "HELLO, WORLD!" at scale 2 with tracking 1?
  q1: null,
  // 2. How many tofu boxes does drawText draw for "Ça va? 👋"?
  q2: null,
  // 3. drawText(frame, "HI", 100, 0, 3, color, { align: "center" }):
  //    which pixel column gets the first ink?
  q3: null,
});


In [ ]:
# Tests + checkpoint for layer 02
!node tests/run.js 02


### In the real world

- Canvas's [`measureText`](https://developer.mozilla.org/en-US/docs/Web/API/CanvasRenderingContext2D/measureText) and [`textAlign`](https://developer.mozilla.org/en-US/docs/Web/API/CanvasRenderingContext2D/textAlign) do what you just did, for outline fonts, with kerning and shaping on top.
- Text shaping engines like [HarfBuzz](https://harfbuzz.github.io/) handle what our font ignores: kerning, ligatures, scripts that join letters, and right-to-left text.
- In Motion Studio's `KineticIntro.tsx`, each word is centered with flexbox and drawn by the browser at a 150 px font size.

### Going further (optional)

- **Design your own glyphs.** Add lowercase letters or symbols to a copy of the font. What's the smallest grid where lowercase "g" still has a readable descender?
- **Outlines.** Draw text with a 1-cell dark outline by drawing it shifted in 8 directions, then on top. Count how many `fillRect` calls that costs, and think about why real renderers don't do it this way.
